# M4 — block_dct_8x8

Non-overlapping 8×8 block DCT, K=16 deterministic zigzag channels, Train-only per-channel normalization, and a 28×28 local grid.

Each run uses the same Stage 0 manifest/cache and verifies the same Stage 1 checkpoint SHA. Candidate code contains no self-promotion rule.



In [1]:
# Edit only paths that cannot be resolved uniquely under /kaggle/input.
STAGE0_RESOURCE_DIR = "/kaggle/input/datasets/maithanhphuong/smartface-pad-lcc-scrfd-cache-v1/stage0"
STAGE1_RESOURCE_DIR = "/kaggle/input/datasets/maithanhphuong/smartface-pad-micro-stage1-v1/stage1_shared_micro_e1"
CELEBA_ROOT = "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
LCC_FASD_ROOT = "/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
CANDIDATE_ID = "M4"
OUTPUT_ROOT = "/kaggle/working/pad_micro_search_v1"
SEED = 42
DIAGNOSTIC_SHUFFLE_SEED = 20260929
BATCH_SIZE = 128
NUM_WORKERS = 4
AMP = True


In [2]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights

CELEBA_MEAN = np.asarray([0.5931, 0.4690, 0.4229], dtype=np.float32)
CELEBA_STD = np.asarray([0.2471, 0.2214, 0.2157], dtype=np.float32)
INPUT_SIZE = 224
E1_ARCHITECTURE = "MobileNetV3-Small-GAP-576-256-BN-HSwish-Dropout02-128-3-v1"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_unique_file(name, root="/kaggle/input", explicit_dir=""):
    base = Path(explicit_dir).expanduser() if explicit_dir else Path(root)
    if explicit_dir and base.is_file():
        matches = [base.resolve()] if base.name == name else []
    else:
        matches = sorted(p.resolve() for p in base.rglob(name) if p.is_file()) if base.exists() else []
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {name}, found {len(matches)}. Set an explicit directory. "
            f"Examples: {[str(p) for p in matches[:8]]}"
        )
    return matches[0]

def find_stage0_dir():
    if STAGE0_RESOURCE_DIR:
        root = Path(STAGE0_RESOURCE_DIR).expanduser().resolve()
        if not (root / "stage0_summary.json").is_file():
            if (root / "stage0" / "stage0_summary.json").is_file():
                root = root / "stage0"
            else:
                raise FileNotFoundError(f"stage0_summary.json not under {root}")
        return root
    matches = sorted(p.parent.resolve() for p in Path("/kaggle/input").rglob("stage0_summary.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one Stage 0 resource directory, found {len(matches)}; set STAGE0_RESOURCE_DIR")
    return matches[0]

def find_celeba_root():
    if CELEBA_ROOT:
        root = Path(CELEBA_ROOT).expanduser().resolve()
        if not (root / "metas/intra_test/train_label.json").is_file():
            raise FileNotFoundError(root / "metas/intra_test/train_label.json")
        return root
    matches = []
    for p in Path("/kaggle/input").rglob("train_label.json"):
        if p.as_posix().endswith("metas/intra_test/train_label.json"):
            matches.append(p.parent.parent.parent.resolve())
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CelebA-Spoof root, found {len(matches)}; set CELEBA_ROOT")
    return matches[0]

def find_lcc_root(manifest_path):
    if LCC_FASD_ROOT:
        root = Path(LCC_FASD_ROOT).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    with open(manifest_path, newline="", encoding="utf-8") as f:
        first = next(csv.DictReader(f))
    rel_parts = PurePosixPath(first["path"]).parts
    matches = []
    for p in Path("/kaggle/input").rglob(rel_parts[-1]):
        if p.is_file() and tuple(p.parts[-len(rel_parts):]) == rel_parts:
            matches.append(p.resolve().parents[len(rel_parts) - 1])
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely resolve LCC root from {first['path']}; set LCC_FASD_ROOT")
    return matches[0]

CELEBA_BBOX_STATUS_FACTORS = {
    "VALID": 1.55,
    "FALLBACK_DET_FAIL": 1.50,
    "FALLBACK_SUSPICIOUS": 1.50,
}


def validate_bbox_record(record, domain, key):
    if not isinstance(record, dict):
        raise TypeError("BBox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if domain == "celeba":
        if status == "INVALID":
            raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
        if status not in CELEBA_BBOX_STATUS_FACTORS:
            raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
        expected_factor = CELEBA_BBOX_STATUS_FACTORS[status]
    elif domain == "lcc":
        if status != "VALID":
            raise RuntimeError("Only Stage 0 LCC VALID records may be scored: {} ({})".format(key, status))
        expected_factor = 1.55
    else:
        raise ValueError("Unknown PAD crop domain: {}".format(domain))
    if "crop_factor" not in record:
        raise KeyError("BBox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    if not np.isfinite(crop_factor) or not np.isclose(crop_factor, expected_factor, rtol=0.0, atol=1e-6):
        raise ValueError(
            "BBox status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected_factor
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return crop_factor


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def labels_from_attack_codes(codes):
    codes = np.asarray(codes, dtype=np.int64)
    labels = np.full(codes.shape, 2, dtype=np.int64)
    labels[codes == 0] = 0
    labels[np.isin(codes, np.arange(1, 8))] = 1
    return labels

def load_stage0_resources():
    stage0 = find_stage0_dir()
    manifest_path = stage0 / "micro_manifest_v1_seed42.npz"
    bbox_path = stage0 / "celeba_micro_bbox_cache.json"
    lcc_manifest_path = stage0 / "lcc_external_dev_manifest.csv"
    lcc_cache_path = stage0 / "lcc_scrfd_cache_micro_v1.json"
    fingerprints_path = stage0 / "micro_resource_fingerprints.json"
    summary_path = stage0 / "stage0_summary.json"
    for path in [manifest_path, bbox_path, lcc_manifest_path, lcc_cache_path, fingerprints_path, summary_path]:
        if not path.is_file():
            raise FileNotFoundError("Stage 0 resource missing: {}".format(path))
    fingerprints = json.loads(fingerprints_path.read_text())
    stage0_summary = json.loads(summary_path.read_text())
    if stage0_summary.get("schema") != "pad_micro_stage0_summary_v1":
        raise RuntimeError("Unsupported Stage 0 summary schema")
    for path, key in [
        (manifest_path, "micro_manifest_sha256"),
        (lcc_manifest_path, "lcc_manifest_sha256"),
        (lcc_cache_path, "lcc_cache_sha256"),
        (bbox_path, "celeba_bbox_cache_sha256"),
    ]:
        if sha256_file(path) != fingerprints.get(key):
            raise RuntimeError("Stage 0 SHA256 mismatch for {}".format(key))
    with np.load(manifest_path, allow_pickle=False) as pack:
        train_keys = pack["micro_train_keys"].astype(str)
        val_keys = pack["micro_val_keys"].astype(str)
        train_labels = pack["micro_train_labels"].astype(np.int64)
        val_labels = pack["micro_val_labels"].astype(np.int64)
        train_codes = pack["micro_train_attack_codes"].astype(np.int64)
        val_codes = pack["micro_val_attack_codes"].astype(np.int64)
    if len(train_keys) != 5000 or len(val_keys) != 2000:
        raise RuntimeError("Stage 0 must freeze exactly 5,000 Train and 2,000 Val keys")
    if len(set(train_keys) & set(val_keys)):
        raise RuntimeError("Stage 0 Train/Val keys overlap")
    for name, labels, codes in [
        ("Train", train_labels, train_codes),
        ("Val", val_labels, val_codes),
    ]:
        if labels.shape != codes.shape or set(map(int, labels)) != {0, 1, 2}:
            raise RuntimeError("Stage 0 {} label/code arrays are malformed or miss a class".format(name))
        if not np.array_equal(labels, labels_from_attack_codes(codes)):
            raise RuntimeError("Stage 0 {} labels do not match E1 v5.3 attack-code mapping".format(name))
    bbox_payload = json.loads(bbox_path.read_text())
    bbox_cache = bbox_payload.get("records", {})
    selected_keys = set(map(str, train_keys)) | set(map(str, val_keys))
    if bbox_payload.get("schema") != "celeba_micro_bbox_cache_v1" or set(bbox_cache) != selected_keys:
        raise RuntimeError("Stage 0 CelebA bbox cache keys/schema differ from the frozen micro manifest")
    for key, record in bbox_cache.items():
        validate_bbox_record(record, "celeba", key)
    lcc_df = pd.read_csv(lcc_manifest_path)
    lcc_payload = json.loads(lcc_cache_path.read_text())
    lcc_records = lcc_payload.get("records", {})
    if lcc_payload.get("schema") != "pad_micro_lcc_scrfd_cache_v1":
        raise RuntimeError("Unsupported Stage 0 LCC cache schema")
    if lcc_df["path"].astype(str).duplicated().any():
        raise RuntimeError("Stage 0 LCC manifest contains duplicate paths")
    if not set(lcc_df["label"].astype(int)).issubset({0, 1}):
        raise RuntimeError("Stage 0 LCC labels must be exactly 0=Real or 1=Attack")
    record_by_path = {}
    for record in lcc_records.values():
        path = str(record.get("path", ""))
        record_by_path.setdefault(path, []).append(record)
    for row in lcc_df.itertuples(index=False):
        matches = record_by_path.get(str(row.path), [])
        if len(matches) != 1:
            raise RuntimeError("Stage 0 LCC cache join for {} has {} rows".format(row.path, len(matches)))
        record = matches[0]
        if str(record.get("status", "")) != str(row.status):
            raise RuntimeError("Stage 0 LCC manifest/cache status mismatch for {}".format(row.path))
        if int(record.get("label", -1)) != int(row.label):
            raise RuntimeError("Stage 0 LCC manifest/cache label mismatch for {}".format(row.path))
        if str(record.get("frame_id", "")) != str(row.record_id):
            raise RuntimeError("Stage 0 LCC manifest record_id mismatch for {}".format(row.path))
        if str(row.status) == "VALID":
            validate_bbox_record(record, "lcc", row.path)
    valid_paths = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    valid_labels = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "label"].astype(int).tolist()
    if not valid_paths or set(valid_labels) != {0, 1}:
        raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
    if len(lcc_df) != int(fingerprints.get("lcc_candidate_n", -1)):
        raise RuntimeError("Stage 0 LCC candidate count mismatch")
    if len(valid_paths) != int(fingerprints.get("lcc_valid_face_n", -1)):
        raise RuntimeError("Stage 0 LCC valid-face count mismatch")
    valid_path_sha = ordered_path_fingerprint(valid_paths)
    if valid_path_sha != fingerprints.get("lcc_valid_path_fingerprint_sha256"):
        raise RuntimeError("Stage 0 ordered LCC VALID path fingerprint mismatch")
    if stage0_summary.get("lcc_valid_path_fingerprint_sha256") != valid_path_sha:
        raise RuntimeError("Stage 0 summary ordered LCC VALID path fingerprint mismatch")
    return {
        "stage0_dir": stage0,
        "train_keys": train_keys,
        "val_keys": val_keys,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_codes": train_codes,
        "val_codes": val_codes,
        "bbox_cache": bbox_cache,
        "lcc_df": lcc_df,
        "lcc_cache": lcc_records,
        "lcc_valid_paths": valid_paths,
        "fingerprints": fingerprints,
    }
def assert_raw_image_paths(resources, celeba_root, lcc_root):
    celeba_keys = list(map(str, resources["train_keys"])) + list(map(str, resources["val_keys"]))
    lcc_paths = resources["lcc_df"]["path"].astype(str).tolist()
    missing_celeba = [str(Path(celeba_root) / key) for key in celeba_keys if not (Path(celeba_root) / key).is_file()]
    missing_lcc = [str(Path(lcc_root) / path) for path in lcc_paths if not (Path(lcc_root) / path).is_file()]
    if missing_celeba or missing_lcc:
        raise FileNotFoundError(
            "Raw PAD image preflight failed before training: CelebA missing={}, LCC missing={}".format(
                missing_celeba[:5], missing_lcc[:5]
            )
        )
    print("Verified raw image paths before training: CelebA selected={}, LCC candidates={}".format(
        len(celeba_keys), len(lcc_paths)
    ))


def runtime_crop_bgr(image, bbox_xyxy, factor):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")
    extent = max(bw, bh) * float(factor)
    side = int(extent)
    cx, cy = x1 + bw / 2.0, y1 + bh / 2.0
    xs, ys = int(cx - extent / 2.0), int(cy - extent / 2.0)
    xa, ya, xb, yb = max(0, xs), max(0, ys), min(w, xs + side), min(h, ys + side)
    if xb <= xa or yb <= ya:
        raise RuntimeError("Empty crop intersection")
    crop = image[ya:yb, xa:xb]
    return cv2.copyMakeBorder(
        crop, max(0, -ys), max(0, ys + side - h),
        max(0, -xs), max(0, xs + side - w), cv2.BORDER_REFLECT_101
    )

def resize_letterbox_rgb(rgb, size=224):
    h, w = rgb.shape[:2]
    ratio = size / max(h, w)
    nh, nw = max(1, int(h * ratio)), max(1, int(w * ratio))
    interp = cv2.INTER_LANCZOS4 if ratio > 1 else cv2.INTER_AREA
    image = cv2.resize(rgb, (nw, nh), interpolation=interp)
    dh, dw = size - nh, size - nw
    return cv2.copyMakeBorder(image, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT_101)

def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb

def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]

def rgb_tensor(rgb):
    unit = rgb.astype(np.float32) / 255.0
    unit = (unit - CELEBA_MEAN) / CELEBA_STD
    return torch.from_numpy(np.ascontiguousarray(unit.transpose(2, 0, 1), dtype=np.float32))

def optional_crop_index(stage0_dir):
    path = Path(stage0_dir) / "crop_cache" / "index.json"
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rgb_crop(image_path, record, domain, key, crop_index, stage0_dir, augment=False):
    crop_factor = validate_bbox_record(record, domain, key)
    cache_key = "{}::{}".format(domain, key)
    if not augment and cache_key in crop_index:
        path = Path(stage0_dir) / "crop_cache" / crop_index[cache_key]
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError("Optional Stage 0 crop cache entry is unreadable: {}".format(path))
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(image_path)
    box = bbox_jitter(record) if augment else record["bbox_xyxy"]
    crop = runtime_crop_bgr(bgr, box, crop_factor)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    if augment:
        rgb = apply_e1_mild_augmentation(rgb)
    return resize_letterbox_rgb(rgb, INPUT_SIZE)
class CelebAMicroDataset(Dataset):
    def __init__(self, keys, labels, codes, root, bbox_records, stage0_dir, train=False):
        self.keys = list(map(str, keys))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.codes = np.asarray(codes, dtype=np.int64)
        self.root = Path(root)
        self.records = bbox_records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.train = bool(train)
        if len(self.keys) != len(self.labels) or len(self.keys) != len(self.codes):
            raise ValueError("CelebA key/label/code lengths differ")
        missing = [k for k in self.keys if k not in self.records]
        if missing:
            raise RuntimeError(f"{len(missing)} CelebA keys lack Stage 0 bbox records")
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop(
            self.root / key, self.records[key], "celeba", key,
            self.crop_index, self.stage0_dir, augment=self.train
        )
        return rgb_tensor(rgb), int(self.labels[i]), key, int(self.codes[i])

class LCCValidDataset(Dataset):
    def __init__(self, frame, root, records, stage0_dir):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.records = records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.rows = []
        by_path = {}
        for rec in records.values():
            by_path.setdefault(str(rec.get("path", "")), []).append(rec)
        for _, row in self.frame.iterrows():
            if str(row["status"]) != "VALID":
                continue
            path = str(row["path"])
            candidates = by_path.get(path, [])
            if len(candidates) != 1:
                raise RuntimeError(f"LCC cache join for {path} has {len(candidates)} records")
            self.rows.append((path, int(row["label"]), candidates[0]))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        key, label, record = self.rows[i]
        rgb = read_rgb_crop(
            self.root / key, record, "lcc", key, self.crop_index,
            self.stage0_dir, augment=False
        )
        return rgb_tensor(rgb), int(label), key, record.get("status", "VALID")

class SpatialE1(nn.Module):
    def __init__(self, imagenet_state=None):
        super().__init__()
        base = models.mobilenet_v3_small(weights=None)
        if imagenet_state is not None:
            missing, unexpected = base.load_state_dict(imagenet_state, strict=False)
            allowed_missing = [k for k in missing if k.startswith("classifier.")]
            bad_missing = [k for k in missing if k not in allowed_missing]
            allowed_unexpected = [k for k in unexpected if k.startswith("classifier.")]
            bad_unexpected = [k for k in unexpected if k not in allowed_unexpected]
            if bad_missing or bad_unexpected:
                raise RuntimeError(
                    "ImageNet MobileNetV3-Small state_dict is incomplete/incompatible; "
                    "backbone missing={}, unexpected={}".format(bad_missing[:5], bad_unexpected[:5])
                )
        self.features = base.features
        self.avgpool = base.avgpool
        self.projection = nn.Sequential(
            nn.Linear(576, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),
        )
    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.projection(x)
    def forward(self, x):
        feature = self.forward_features(x)
        return self.classifier(feature)

def load_imagenet_weights(local_path=""):
    if local_path:
        state = torch.load(local_path, map_location="cpu", weights_only=False)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        if not isinstance(state, dict):
            raise RuntimeError("Local ImageNet weights must be a state_dict")
        return state
    try:
        return MobileNet_V3_Small_Weights.DEFAULT.get_state_dict(progress=True)
    except Exception as exc:
        raise RuntimeError(
            "ImageNet MobileNetV3-Small weights are unavailable. Enable Kaggle Internet "
            "or set IMAGENET_WEIGHTS_PATH to a compatible torchvision state_dict. "
            "Random initialization is intentionally not used."
        ) from exc

def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)
def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))
def threshold_from_val(y3, scores):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("Threshold calibration needs non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("CelebA labels must be 0=Real, 1=Physical, 2=Digital")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("Threshold calibration requires at least one Real and one Attack sample")
    if not np.isfinite(scores).all():
        raise ValueError("Threshold calibration scores contain NaN or Infinity")
    values = np.unique(scores)
    lower = np.nextafter(values[0], -np.inf)
    upper = np.nextafter(values[-1], np.inf)
    if not np.isfinite(lower):
        lower = values[0]
    if not np.isfinite(upper):
        upper = values[-1]
    midpoints = values[:-1] / 2.0 + values[1:] / 2.0
    candidates = np.concatenate(([lower], midpoints, [upper]))
    best = None
    for threshold in candidates:
        pred_real = scores >= threshold
        real = y_real == 1
        attack = y_real == 0
        bpcer = float(np.mean(~pred_real[real]))
        apcer = float(np.mean(pred_real[attack]))
        acer = 0.5 * (apcer + bpcer)
        key = (acer, abs(float(threshold)), float(threshold))
        if best is None or key < best[0]:
            best = (key, float(threshold))
    if best is None or not np.isfinite(best[1]):
        raise RuntimeError("Threshold search did not produce a finite threshold")
    return best[1]
def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }
def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores

def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result

def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes
def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    logits = np.concatenate(logits_all) if logits_all else np.empty((0, 3), dtype=np.float32)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy() if len(logits) else np.empty(0)
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0] if len(logits) else [],
        "logit_physical": logits[:, 1] if len(logits) else [],
        "logit_digital": logits[:, 2] if len(logits) else [],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(scored, on=["path", "label"], how="left", validate="one_to_one")
    valid = full["status"].eq("VALID")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full

def trainable_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


In [3]:
import json
import math
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

CANDIDATE_CONFIG = {
    "M0": {"name": "global_dct_gap", "input_channels": 1, "pool_grid": 1, "representation": "global 224x224 luminance DCT; signed-log; per-sample z-score"},
    "M1": {"name": "dct_pool4x4", "input_channels": 1, "pool_grid": 4, "representation": "same as M0; AdaptiveAvgPool2d(4x4) before 64-D projection"},
    "M2": {"name": "coord_dct_pool4x4", "input_channels": 3, "pool_grid": 4, "representation": "normalized DCT + fixed U and V coordinate maps"},
    "M3": {"name": "band_aware_dct", "input_channels": 3, "pool_grid": 4, "representation": "low/mid/high fixed radial masks on normalized DCT"},
    "M4": {"name": "block_dct_8x8", "input_channels": 16, "pool_grid": 4, "representation": "8x8 block DCT, K=16 zigzag channels, Train-only per-channel standardization"},
}
if CANDIDATE_ID not in CANDIDATE_CONFIG:
    raise ValueError(f"Unknown candidate: {CANDIDATE_ID}")
CANDIDATE = CANDIDATE_CONFIG[CANDIDATE_ID]
CANDIDATE_NAME = CANDIDATE["name"]
OUT_DIR_NAME = f"{CANDIDATE_ID}_{CANDIDATE_NAME}"

def find_stage1_dir():
    if STAGE1_RESOURCE_DIR:
        root = Path(STAGE1_RESOURCE_DIR).expanduser().resolve()
        if (root / "micro_e1_v1_best.pth").is_file():
            return root
        matches = sorted(p.parent.resolve() for p in root.rglob("micro_e1_v1_best.pth"))
    else:
        matches = sorted(p.parent.resolve() for p in Path("/kaggle/input").rglob("micro_e1_v1_best.pth"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one Stage 1 checkpoint directory, found {len(matches)}; set STAGE1_RESOURCE_DIR")
    return matches[0]

def load_stage1_checkpoint(stage1_dir, expected_resources):
    path = Path(stage1_dir) / "micro_e1_v1_best.pth"
    meta_path = Path(stage1_dir) / "micro_e1_v1_meta.json"
    if not path.is_file() or not meta_path.is_file():
        raise FileNotFoundError("Stage 1 checkpoint bundle is incomplete under {}".format(stage1_dir))
    meta = json.loads(meta_path.read_text())
    if meta.get("schema") != "pad_micro_stage1_meta_v1":
        raise RuntimeError("Unsupported Stage 1 metadata schema")
    actual_sha = sha256_file(path)
    if actual_sha != meta.get("checkpoint_sha256"):
        raise RuntimeError("Stage 1 checkpoint SHA256 does not match its metadata")
    if meta.get("resource_hashes") != expected_resources:
        raise RuntimeError("Stage 1 resource fingerprints differ from this Stage 0 bundle")
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(checkpoint, dict) or not isinstance(checkpoint.get("state_dict"), dict):
        raise RuntimeError("Stage 1 checkpoint does not contain the required state_dict")
    return checkpoint, meta, actual_sha
class DSBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=2):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False)
        self.depth_bn = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.point_bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.act(self.depth_bn(self.depthwise(x)))
        return self.act(self.point_bn(self.pointwise(x)))

class FrequencyEncoder(nn.Module):
    def __init__(self, candidate_id, in_channels, pool_grid):
        super().__init__()
        if candidate_id == "M4":
            self.bottleneck = nn.Sequential(
                nn.Conv2d(in_channels, 32, 1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )
            self.stem = nn.Sequential(
                nn.Conv2d(32, 32, 3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )
        else:
            self.bottleneck = nn.Identity()
            self.stem = nn.Sequential(
                nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )
        self.block1 = DSBlock(32, 32, stride=2)
        self.block2 = DSBlock(32, 64, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((pool_grid, pool_grid))
        self.project = nn.Sequential(nn.Linear(64 * pool_grid * pool_grid, 64), nn.ReLU(inplace=True))
    def forward(self, x):
        x = self.bottleneck(x)
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x).flatten(1)
        return self.project(x)

class FrequencyOnly(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(64, 128), nn.Hardswish(), nn.Dropout(0.2), nn.Linear(128, 3)
        )
    def forward(self, frequency):
        return self.classifier(self.encoder(frequency))

class ConcatScreen(nn.Module):
    def __init__(self, spatial, frequency_encoder):
        super().__init__()
        self.spatial = spatial
        self.frequency_encoder = frequency_encoder
        self.fusion = nn.Sequential(
            nn.Linear(320, 128), nn.Hardswish(), nn.Dropout(0.2), nn.Linear(128, 3)
        )
    def forward_features(self, x, frequency):
        with torch.no_grad():
            spatial_feature = self.spatial.forward_features(x)
        frequency_feature = self.frequency_encoder(frequency)
        return spatial_feature, frequency_feature
    def forward(self, x, frequency):
        spatial_feature, frequency_feature = self.forward_features(x, frequency)
        return self.fusion(torch.cat([spatial_feature, frequency_feature], dim=1))

def make_dct_basis():
    n = np.arange(8, dtype=np.float64)
    basis = np.zeros((8, 8), dtype=np.float64)
    for u in range(8):
        alpha = math.sqrt(1 / 8) if u == 0 else math.sqrt(2 / 8)
        basis[u] = alpha * np.cos((2 * n + 1) * u * math.pi / 16)
    return basis.astype(np.float32)

def zigzag_coefficients(k=16):
    coords = []
    for diagonal in range(15):
        low = max(0, diagonal - 7)
        high = min(7, diagonal)
        row_values = list(range(high, low - 1, -1)) if diagonal % 2 == 0 else list(range(low, high + 1))
        for u in row_values:
            coords.append((u, diagonal - u))
            if len(coords) == k:
                return coords
    return coords[:k]

DCT_BASIS = make_dct_basis()
M4_COEFFICIENTS = zigzag_coefficients(16)
COORD_U = np.tile(np.linspace(0, 1, 224, dtype=np.float32)[None, :], (224, 1))
COORD_V = np.tile(np.linspace(0, 1, 224, dtype=np.float32)[:, None], (1, 224))
RADIAL = np.sqrt(COORD_U ** 2 + COORD_V ** 2) / np.sqrt(2.0)
BAND_MASKS = np.stack([
    RADIAL < (1.0 / 3.0),
    (RADIAL >= (1.0 / 3.0)) & (RADIAL < (2.0 / 3.0)),
    RADIAL >= (2.0 / 3.0),
]).astype(np.float32)
if BAND_MASKS.shape != (3, 224, 224) or not np.all(BAND_MASKS.sum(axis=0) == 1):
    raise RuntimeError("Band masks must partition every DCT coefficient exactly once")

def luminance_unit(rgb):
    x = rgb.astype(np.float32) / 255.0
    return (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).astype(np.float32)

def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

def global_dct_map(rgb):
    coeff = cv2.dct(np.ascontiguousarray(luminance_unit(rgb)))
    coeff = signed_log(coeff)
    coeff = (coeff - float(coeff.mean())) / max(float(coeff.std()), 1e-6)
    return np.ascontiguousarray(coeff[None, ...], dtype=np.float32)

def block_dct_raw(rgb):
    y = luminance_unit(rgb).astype(np.float32)
    blocks = y.reshape(28, 8, 28, 8).transpose(0, 2, 1, 3)
    coeff = np.einsum("ui,xyij,vj->xyuv", DCT_BASIS, blocks, DCT_BASIS, optimize=True)
    channels = np.stack([coeff[:, :, u, v] for u, v in M4_COEFFICIENTS], axis=0)
    return signed_log(channels).astype(np.float32)

def frequency_map(rgb, m4_mean=None, m4_std=None):
    if CANDIDATE_ID in ("M0", "M1"):
        return global_dct_map(rgb)
    if CANDIDATE_ID in ("M2", "M3"):
        base = global_dct_map(rgb)
    if CANDIDATE_ID == "M2":
        return np.concatenate([base, COORD_U[None], COORD_V[None]], axis=0).astype(np.float32)
    if CANDIDATE_ID == "M3":
        return (base * BAND_MASKS).astype(np.float32)
    if CANDIDATE_ID == "M4":
        if m4_mean is None or m4_std is None:
            raise RuntimeError("M4 Train-only normalization stats have not been initialized")
        mean = np.asarray(m4_mean, dtype=np.float32).reshape(-1, 1, 1)
        std = np.asarray(m4_std, dtype=np.float32).reshape(-1, 1, 1)
        return ((block_dct_raw(rgb) - mean) / np.maximum(std, 1e-6)).astype(np.float32)
    raise ValueError(CANDIDATE_ID)

def estimate_m4_train_stats(keys, root, bbox_records, stage0_dir):
    total = np.zeros(16, dtype=np.float64)
    total_sq = np.zeros(16, dtype=np.float64)
    count = 0
    crop_index = optional_crop_index(stage0_dir)
    keys = list(map(str, keys))
    for i, key in enumerate(keys, start=1):
        rgb = read_rgb_crop(root / key, bbox_records[key], "celeba", key, crop_index, stage0_dir, augment=False)
        values = block_dct_raw(rgb).astype(np.float64)
        total += values.sum(axis=(1, 2))
        total_sq += np.square(values).sum(axis=(1, 2))
        count += values.shape[1] * values.shape[2]
        if i % 1000 == 0:
            print("M4 Train-only DCT statistics: {}/{}".format(i, len(keys)))
    if count != len(keys) * 28 * 28 or count <= 0:
        raise RuntimeError("M4 normalization did not consume exactly the deterministic Train base crops")
    mean = total / count
    variance = np.maximum(total_sq / count - mean ** 2, 0.0)
    std = np.maximum(np.sqrt(variance), 1e-6)
    if not np.isfinite(mean).all() or not np.isfinite(std).all() or np.any(std <= 0):
        raise RuntimeError("M4 Train-only normalization statistics are non-finite or non-positive")
    return mean.astype(np.float32), std.astype(np.float32)
class CandidateCelebADataset(Dataset):
    def __init__(self, keys, labels, codes, root, bbox_records, stage0_dir, train=False, m4_mean=None, m4_std=None):
        self.keys = list(map(str, keys))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.codes = np.asarray(codes, dtype=np.int64)
        self.root = Path(root)
        self.records = bbox_records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.train = bool(train)
        self.m4_mean, self.m4_std = m4_mean, m4_std
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop(
            self.root / key, self.records[key], "celeba", key,
            self.crop_index, self.stage0_dir, augment=self.train
        )
        return rgb_tensor(rgb), torch.from_numpy(frequency_map(rgb, self.m4_mean, self.m4_std)), int(self.labels[i]), key, int(self.codes[i])

class CandidateLCCDataset(Dataset):
    def __init__(self, frame, root, records, stage0_dir, m4_mean=None, m4_std=None):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.m4_mean, self.m4_std = m4_mean, m4_std
        by_path = {}
        for record in records.values():
            by_path.setdefault(str(record.get("path", "")), []).append(record)
        self.rows = []
        for _, row in self.frame.iterrows():
            if str(row["status"]) != "VALID":
                continue
            path = str(row["path"])
            matches = by_path.get(path, [])
            if len(matches) != 1:
                raise RuntimeError(f"LCC cache join for {path} has {len(matches)} records")
            self.rows.append((path, int(row["label"]), matches[0]))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        key, label, record = self.rows[i]
        rgb = read_rgb_crop(self.root / key, record, "lcc", key, self.crop_index, self.stage0_dir, augment=False)
        return rgb_tensor(rgb), torch.from_numpy(frequency_map(rgb, self.m4_mean, self.m4_std)), int(label), key, int(label)

def candidate_frequency_encoder():
    return FrequencyEncoder(CANDIDATE_ID, CANDIDATE["input_channels"], CANDIDATE["pool_grid"])

def model_logits_frequency(model, loader, device, amp):
    model.eval()
    logits_all, labels, keys, codes = [], [], [], []
    with torch.no_grad():
        for _x, freq, y, batch_keys, batch_codes in loader:
            freq = freq.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(freq)
            logits_all.append(logits.float().cpu().numpy())
            labels.extend(np.asarray(y, dtype=np.int64).tolist())
            keys.extend(list(batch_keys))
            codes.extend(np.asarray(batch_codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Frequency evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(labels, dtype=np.int64)
    codes = np.asarray(codes, dtype=np.int64)
    if len(logits) != len(labels) or len(keys) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Frequency evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys)), codes
def model_logits_concat(model, loader, device, amp, return_features=False):
    model.eval()
    logits_all, labels, keys, codes = [], [], [], []
    spatial_all, frequency_all = [], []
    with torch.no_grad():
        for x, freq, y, batch_keys, batch_codes in loader:
            x = x.to(device, non_blocking=True)
            freq = freq.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                if return_features:
                    spatial, frequency = model.forward_features(x, freq)
                    logits = model.fusion(torch.cat([spatial, frequency], dim=1))
                    spatial_all.append(spatial.float().cpu())
                    frequency_all.append(frequency.float().cpu())
                else:
                    logits = model(x, freq)
            logits_all.append(logits.float().cpu().numpy())
            labels.extend(np.asarray(y, dtype=np.int64).tolist())
            keys.extend(list(batch_keys))
            codes.extend(np.asarray(batch_codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("CONCAT evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(labels, dtype=np.int64)
    codes = np.asarray(codes, dtype=np.int64)
    if len(logits) != len(labels) or len(keys) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("CONCAT evaluation outputs have inconsistent sample counts")
    payload = (logits, labels, list(map(str, keys)), codes)
    if return_features:
        spatial = torch.cat(spatial_all)
        frequency = torch.cat(frequency_all)
        if len(spatial) != len(labels) or len(frequency) != len(labels):
            raise RuntimeError("CONCAT feature outputs have inconsistent sample counts")
        return payload + (spatial, frequency)
    return payload
def candidate_prediction_frame(keys, y3, codes, logits, threshold):
    return make_prediction_frame(keys, y3, codes, logits, threshold)

def lcc_prediction_frame(lcc_frame, keys, logits):
    keys = list(map(str, keys))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Candidate LCC")
    logits = np.asarray(logits)
    if logits.shape != (len(keys), 3) or not np.isfinite(logits).all():
        raise RuntimeError("LCC logits must be finite with shape [common_valid_n, 3]")
    scored = pd.DataFrame({
        "path": keys,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": binary_score(torch.as_tensor(logits)).cpu().numpy(),
    })
    result = lcc_frame[["path", "label", "status"]].merge(
        scored, on="path", how="left", validate="one_to_one"
    )
    valid = result["status"].astype(str).eq("VALID")
    if int(valid.sum()) != len(keys) or result.loc[valid, "pad_logit_score"].isna().any():
        raise RuntimeError("Candidate inference did not score every common LCC VALID path")
    result["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    return result
def lcc_metrics_and_predictions(frame, keys, logits, threshold):
    result = lcc_prediction_frame(frame, keys, logits)
    valid = result["status"].astype(str).eq("VALID")
    scores = result.loc[valid, "pad_logit_score"].to_numpy(dtype=np.float64)
    labels = result.loc[valid, "label"].to_numpy(dtype=np.int64)
    coverage = float(valid.sum() / max(1, len(result)))
    metrics = metrics_binary(labels, scores, threshold, coverage=coverage, candidate_n=len(result))
    result["locked_logit_threshold"] = float(threshold)
    result["pred_real"] = np.where(valid, result["pad_logit_score"] >= threshold, np.nan)
    result["correct_binary"] = np.where(valid, result["pred_real"] == (result["label"] == 0), np.nan)
    return metrics, result

def train_one_epoch_frequency(model, loader, optimizer, criterion, scaler, device, amp):
    model.train()
    losses = []
    for _x, freq, y, _keys, _codes in loader:
        freq, y = freq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp):
            logits = model(freq)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

def train_one_epoch_concat(model, loader, optimizer, criterion, scaler, device, amp, phase):
    model.train()
    model.spatial.eval()
    if phase == "B1":
        model.frequency_encoder.eval()
    else:
        model.frequency_encoder.train()
    model.spatial.requires_grad_(False)
    losses = []
    for x, freq, y, _keys, _codes in loader:
        x, freq, y = x.to(device, non_blocking=True), freq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp):
            logits = model(x, freq)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        trainable = [p for p in model.parameters() if p.requires_grad]
        torch.nn.utils.clip_grad_norm_(trainable, 5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

def deterministic_derangement(n, seed):
    n = int(n)
    if n < 2:
        raise ValueError("FREQ_SHUFFLE requires at least two evaluation samples")
    rng = np.random.default_rng(int(seed))
    perm = np.arange(n, dtype=np.int64)
    # Sattolo's algorithm creates one cycle and therefore has no fixed points.
    for i in range(n - 1, 0, -1):
        j = int(rng.integers(0, i))
        perm[i], perm[j] = perm[j], perm[i]
    fixed_point_count = int(np.count_nonzero(perm == np.arange(n)))
    if fixed_point_count != 0:
        raise AssertionError("Deterministic shuffle produced fixed points")
    fingerprint = hashlib.sha256(np.asarray(perm, dtype="<i8").tobytes()).hexdigest()
    return perm, fingerprint, fixed_point_count


def branch_diagnostics(model, loader_labels, keys, spatial, frequency, threshold, device, expected_keys):
    y = np.asarray(loader_labels, dtype=np.int64)
    keys = list(map(str, keys))
    assert_ordered_keys(keys, expected_keys, "Branch diagnostics")
    if spatial.ndim != 2 or spatial.shape[1] != 256 or frequency.ndim != 2 or frequency.shape[1] != 64:
        raise RuntimeError("Branch diagnostic features must be [N,256] spatial and [N,64] frequency")
    if len(y) != len(keys) or len(y) != len(spatial) or len(y) != len(frequency):
        raise RuntimeError("Branch diagnostics labels/features/keys are not aligned")
    if not torch.isfinite(spatial).all() or not torch.isfinite(frequency).all():
        raise RuntimeError("Branch diagnostic features contain NaN or Infinity")
    perm, perm_sha256, fixed_point_count = deterministic_derangement(len(y), DIAGNOSTIC_SHUFFLE_SEED)
    shuffled = frequency[torch.as_tensor(perm, dtype=torch.long)]
    modes = {
        "FULL": torch.cat([spatial, frequency], dim=1),
        "FREQ_ZERO": torch.cat([spatial, torch.zeros_like(frequency)], dim=1),
        "FREQ_SHUFFLE": torch.cat([spatial, shuffled], dim=1),
        "SPATIAL_ZERO": torch.cat([torch.zeros_like(spatial), frequency], dim=1),
    }
    results, rows = {}, []
    with torch.no_grad():
        for mode, joined in modes.items():
            mode_logits = model.fusion(joined.to(device)).float().cpu().numpy()
            score = binary_score(torch.as_tensor(mode_logits)).cpu().numpy()
            metric = metrics_binary(y, score, threshold, coverage=1.0, candidate_n=len(y))
            results[mode] = metric
            rows.append({"mode": mode, **metric})
    first = model.fusion[0]
    ws = first.weight[:, :256].detach().float()
    wf = first.weight[:, 256:].detach().float()
    spatial_contrib = torch.nn.functional.linear(spatial, first.weight[:, :256].detach().cpu())
    frequency_contrib = torch.nn.functional.linear(frequency, first.weight[:, 256:].detach().cpu())
    norms = {
        "spatial_feature_l2_mean": float(torch.linalg.vector_norm(spatial, dim=1).mean()),
        "spatial_feature_l2_std": float(torch.linalg.vector_norm(spatial, dim=1).std(unbiased=False)),
        "frequency_feature_l2_mean": float(torch.linalg.vector_norm(frequency, dim=1).mean()),
        "frequency_feature_l2_std": float(torch.linalg.vector_norm(frequency, dim=1).std(unbiased=False)),
        "fusion_spatial_weight_slice_frobenius_norm": float(torch.linalg.vector_norm(ws)),
        "fusion_frequency_weight_slice_frobenius_norm": float(torch.linalg.vector_norm(wf)),
        "spatial_linear_contribution_l2_mean": float(torch.linalg.vector_norm(spatial_contrib, dim=1).mean()),
        "frequency_linear_contribution_l2_mean": float(torch.linalg.vector_norm(frequency_contrib, dim=1).mean()),
        "diagnostic_shuffle_seed": int(DIAGNOSTIC_SHUFFLE_SEED),
        "permutation_sha256": perm_sha256,
        "fixed_point_count": fixed_point_count,
        "permutation_n": int(len(perm)),
    }
    numeric_norms = [value for key, value in norms.items() if key.endswith("mean") or key.endswith("std") or "frobenius" in key or "contribution" in key]
    if not np.isfinite(np.asarray(numeric_norms, dtype=np.float64)).all():
        raise RuntimeError("Branch feature/contribution diagnostics contain NaN or Infinity")
    return results, rows, norms
def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def make_candidate_summary(resources, stage1_sha, freq_val, freq_lcc, concat_val, concat_lcc, ablation, diagnostics, params):
    return {
        "schema": "pad_micro_candidate_summary_v1",
        "lcc_dataset_role": "LCC-FASD external-dev / cross-domain stress set",
        "candidate_id": CANDIDATE_ID,
        "candidate_name": CANDIDATE_NAME,
        "seed": int(SEED),
        "budget": {"train_n": int(len(resources["train_keys"])), "val_n": int(len(resources["val_keys"]))},
        "resource_hashes": {
            **resources["fingerprints"],
            "shared_micro_e1_checkpoint_sha256": stage1_sha,
        },
        "frequency_only": {"val": freq_val, "lcc": freq_lcc},
        "concat": {"val": concat_val, "lcc": concat_lcc},
        "branch_ablation": ablation,
        "feature_diagnostics": diagnostics,
        "params": params,
    }

def zip_result_dir(folder, output):
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(output).resolve():
                archive.write(path, arcname=Path(folder).name / path.relative_to(folder))

seed_everything(SEED)
resources = load_stage0_resources()
STAGE0_DIR = resources["stage0_dir"]
celeba_root = find_celeba_root()
lcc_root = find_lcc_root(STAGE0_DIR / "lcc_external_dev_manifest.csv")
assert_raw_image_paths(resources, celeba_root, lcc_root)
stage1_dir = find_stage1_dir()
stage1_checkpoint, stage1_meta, stage1_sha = load_stage1_checkpoint(stage1_dir, resources["fingerprints"])
if len(resources["train_keys"]) != 5000 or len(resources["val_keys"]) != 2000:
    raise RuntimeError("Candidate requires the shared 5K/2K micro manifest")
lcc_frame = resources["lcc_df"]
lcc_candidate_n = len(lcc_frame)
lcc_valid_n = int((lcc_frame["status"].astype(str) == "VALID").sum())
lcc_coverage = lcc_valid_n / max(1, lcc_candidate_n)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Stage 2 training requires a Kaggle GPU accelerator")
use_amp = bool(AMP and device.type == "cuda")
encoder = FrequencyEncoder(CANDIDATE_ID, CANDIDATE["input_channels"], CANDIDATE["pool_grid"])
frequency_model = FrequencyOnly(encoder).to(device)
print("=== MICRO SEARCH PREFLIGHT ===")
print("candidate:", CANDIDATE_ID, CANDIDATE_NAME)
print("Stage 0 resource directory:", STAGE0_DIR)
print("Stage 1 result directory:", stage1_dir)
print("Stage 1 checkpoint path/SHA:", stage1_dir / "micro_e1_v1_best.pth", stage1_sha)
print("seed / diagnostic shuffle seed:", SEED, DIAGNOSTIC_SHUFFLE_SEED)
print("train_n / val_n:", len(resources["train_keys"]), "/", len(resources["val_keys"]))
print("train class counts:", dict(Counter(map(int, resources["train_labels"]))))
print("val class counts:", dict(Counter(map(int, resources["val_labels"]))))
print("parent E1 manifest SHA:", resources["fingerprints"]["parent_e1_manifest_sha256"])
print("micro manifest SHA:", resources["fingerprints"]["micro_manifest_sha256"])
print("LCC manifest/cache SHA:", resources["fingerprints"]["lcc_manifest_sha256"], resources["fingerprints"]["lcc_cache_sha256"])
print("LCC candidates/common VALID:", lcc_candidate_n, lcc_valid_n)
print("LCC common VALID path SHA:", resources["fingerprints"]["lcc_valid_path_fingerprint_sha256"])
print("CelebA root:", celeba_root)
print("LCC root/cache:", lcc_root, STAGE0_DIR / "lcc_scrfd_cache_micro_v1.json")
print("device / AMP:", device, use_amp)
print("Phase A trainable params:", sum(p.numel() for p in frequency_model.parameters() if p.requires_grad))

m4_mean = m4_std = None
out_dir = Path(OUTPUT_ROOT) / OUT_DIR_NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
if CANDIDATE_ID == "M4":
    m4_mean, m4_std = estimate_m4_train_stats(
        resources["train_keys"], celeba_root, resources["bbox_cache"], STAGE0_DIR
    )
    (out_dir / "m4_train_only_normalization_stats.json").write_text(json.dumps({
        "source": "micro-TRAIN only; deterministic base crops; no Val or LCC pixels",
        "coefficient_ids_uv": M4_COEFFICIENTS,
        "mean": m4_mean.tolist(), "std": m4_std.tolist(),
        "micro_train_keys_fingerprint": resources["fingerprints"]["micro_train_keys_fingerprint"],
    }, indent=2, allow_nan=False) + "\n")

train_ds = CandidateCelebADataset(
    resources["train_keys"], resources["train_labels"], resources["train_codes"],
    celeba_root, resources["bbox_cache"], STAGE0_DIR, train=True,
    m4_mean=m4_mean, m4_std=m4_std
)
val_ds = CandidateCelebADataset(
    resources["val_keys"], resources["val_labels"], resources["val_codes"],
    celeba_root, resources["bbox_cache"], STAGE0_DIR, train=False,
    m4_mean=m4_mean, m4_std=m4_std
)
lcc_ds = CandidateLCCDataset(lcc_frame, lcc_root, resources["lcc_cache"], STAGE0_DIR, m4_mean, m4_std)
loader_args = {
    "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "pin_memory": True, "persistent_workers": NUM_WORKERS > 0,
}
train_loader = DataLoader(train_ds, shuffle=True, drop_last=False, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_args)
lcc_loader = DataLoader(lcc_ds, shuffle=False, drop_last=False, **loader_args)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
freq_optimizer = torch.optim.AdamW(frequency_model.parameters(), lr=3e-4, weight_decay=1e-4)
freq_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
freq_history = []
freq_best_acer, freq_best_epoch, freq_best_state = float("inf"), None, None
freq_patience, freq_max_epochs = 2, 5
for epoch in range(1, freq_max_epochs + 1):
    started = time.time()
    train_loss = train_one_epoch_frequency(
        frequency_model, train_loader, freq_optimizer, criterion, freq_scaler, device, use_amp
    )
    val_logits, val_y, val_keys, val_codes = model_logits_frequency(frequency_model, val_loader, device, use_amp)
    val_scores = binary_score(torch.as_tensor(val_logits)).cpu().numpy()
    val_threshold = threshold_from_val(val_y, val_scores)
    val_metrics, _ = metrics_celeba(val_y, val_logits, val_threshold)
    row = {
        "phase": "A", "epoch": epoch, "train_loss": train_loss,
        "val_acer": val_metrics["acer"], "val_auc": val_metrics["auc"],
        "val_threshold": float(val_threshold), "seconds": time.time() - started,
    }
    freq_history.append(row)
    print(row)
    if val_metrics["acer"] < freq_best_acer:
        freq_best_acer, freq_best_epoch = val_metrics["acer"], epoch
        freq_best_state = {k: v.detach().cpu().clone() for k, v in frequency_model.state_dict().items()}
    elif epoch >= 3 and epoch - freq_best_epoch >= freq_patience:
        print("Phase A early stop on micro-Val ACER")
        break
if freq_best_state is None:
    raise RuntimeError("Phase A did not produce a selected checkpoint")
frequency_model.load_state_dict(freq_best_state)
frequency_model.to(device)
freq_val_logits, freq_val_y, freq_val_keys, freq_val_codes = model_logits_frequency(
    frequency_model, val_loader, device, use_amp
)
assert_ordered_keys(freq_val_keys, resources["val_keys"], "Phase A micro-Val")
freq_threshold = threshold_from_val(
    freq_val_y, binary_score(torch.as_tensor(freq_val_logits)).cpu().numpy()
)
freq_val_metrics, _ = metrics_celeba(freq_val_y, freq_val_logits, freq_threshold)
freq_val_predictions = candidate_prediction_frame(
    freq_val_keys, freq_val_y, freq_val_codes, freq_val_logits, freq_threshold
)
freq_lcc_logits, freq_lcc_y, freq_lcc_keys, freq_lcc_codes = model_logits_frequency(
    frequency_model, lcc_loader, device, use_amp
)
assert_ordered_keys(freq_lcc_keys, resources["lcc_valid_paths"], "Phase A LCC")
freq_lcc_scores = binary_score(torch.as_tensor(freq_lcc_logits)).cpu().numpy()
freq_lcc_metrics = metrics_binary(
    freq_lcc_y, freq_lcc_scores, freq_threshold,
    coverage=lcc_coverage, candidate_n=lcc_candidate_n
)
freq_lcc_predictions = lcc_prediction_frame(lcc_frame, freq_lcc_keys, freq_lcc_logits)
freq_lcc_valid = freq_lcc_predictions["status"].astype(str).eq("VALID")
freq_lcc_predictions["locked_logit_threshold"] = freq_threshold
freq_lcc_predictions["pred_real"] = np.where(
    freq_lcc_valid, freq_lcc_predictions["pad_logit_score"] >= freq_threshold, np.nan
)
freq_lcc_predictions["correct_binary"] = np.where(
    freq_lcc_valid, freq_lcc_predictions["pred_real"] == (freq_lcc_predictions["label"] == 0), np.nan
)

freq_best_path = out_dir / "frequency_best.pth"
torch.save({"model_state": freq_best_state, "selected_epoch": int(freq_best_epoch), "validation_acer": float(freq_best_acer)}, freq_best_path)
freq_checkpoint_sha = sha256_file(freq_best_path)
freq_val_predictions.to_csv(out_dir / "frequency_val_predictions.csv", index=False)
freq_lcc_predictions.to_csv(out_dir / "frequency_lcc_predictions.csv", index=False)
pd.DataFrame(freq_history).to_csv(out_dir / "frequency_training_history.csv", index=False)
freq_meta = {
    "schema": "pad_micro_frequency_meta_v1", "candidate_id": CANDIDATE_ID,
    "candidate_name": CANDIDATE_NAME, "checkpoint_sha256": freq_checkpoint_sha,
    "selected_epoch": int(freq_best_epoch), "locked_validation_logit_threshold": float(freq_threshold),
    "threshold_source": "CelebA micro-Val only", "resource_hashes": resources["fingerprints"],
    "frequency_dim": 64, "candidate_config": CANDIDATE,
}
if CANDIDATE_ID == "M4":
    freq_meta["m4_normalization_file"] = "m4_train_only_normalization_stats.json"
(out_dir / "frequency_meta.json").write_text(json.dumps(freq_meta, indent=2, allow_nan=False) + "\n")

# Warm-start Phase B from the one frozen shared spatial checkpoint and Phase A encoder.
spatial = SpatialE1(imagenet_state=None)
spatial.load_state_dict(stage1_checkpoint["state_dict"], strict=True)
freq_for_concat = candidate_frequency_encoder()
freq_for_concat.load_state_dict(frequency_model.encoder.state_dict(), strict=True)
concat_model = ConcatScreen(spatial, freq_for_concat).to(device)
concat_model.spatial.requires_grad_(False)
concat_model.spatial.eval()
concat_model.frequency_encoder.requires_grad_(False)
concat_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
concat_history = []
concat_best_acer, concat_best_epoch, concat_best_state = float("inf"), None, None
for phase, phase_epochs in [("B1", 2), ("B2", 2)]:
    if phase == "B1":
        concat_model.frequency_encoder.requires_grad_(False)
        optimizer = torch.optim.AdamW(concat_model.fusion.parameters(), lr=3e-4, weight_decay=1e-4)
    else:
        concat_model.frequency_encoder.requires_grad_(True)
        optimizer = torch.optim.AdamW(
            [
                {"params": concat_model.frequency_encoder.parameters(), "lr": 1e-4},
                {"params": concat_model.fusion.parameters(), "lr": 3e-4},
            ],
            weight_decay=1e-4,
        )
    for phase_epoch in range(1, phase_epochs + 1):
        global_epoch = len(concat_history) + 1
        started = time.time()
        train_loss = train_one_epoch_concat(
            concat_model, train_loader, optimizer, criterion, concat_scaler, device, use_amp, phase
        )
        val_logits, val_y, val_keys, val_codes = model_logits_concat(
            concat_model, val_loader, device, use_amp
        )
        val_threshold = threshold_from_val(
            val_y, binary_score(torch.as_tensor(val_logits)).cpu().numpy()
        )
        val_metrics, _ = metrics_celeba(val_y, val_logits, val_threshold)
        row = {
            "phase": phase, "phase_epoch": phase_epoch, "global_epoch": global_epoch,
            "train_loss": train_loss, "val_acer": val_metrics["acer"],
            "val_auc": val_metrics["auc"], "val_threshold": float(val_threshold),
            "seconds": time.time() - started,
        }
        concat_history.append(row)
        print(row)
        if val_metrics["acer"] < concat_best_acer:
            concat_best_acer, concat_best_epoch = val_metrics["acer"], global_epoch
            concat_best_state = {k: v.detach().cpu().clone() for k, v in concat_model.state_dict().items()}
if concat_best_state is None:
    raise RuntimeError("Phase B did not produce a selected checkpoint")
concat_last_state = {k: v.detach().cpu().clone() for k, v in concat_model.state_dict().items()}
concat_model.load_state_dict(concat_best_state, strict=True)
concat_model.to(device)
concat_model.eval()
concat_val_logits, concat_val_y, concat_val_keys, concat_val_codes, val_spatial, val_frequency = model_logits_concat(
    concat_model, val_loader, device, use_amp, return_features=True
)
assert_ordered_keys(concat_val_keys, resources["val_keys"], "Phase B micro-Val")
concat_threshold = threshold_from_val(
    concat_val_y, binary_score(torch.as_tensor(concat_val_logits)).cpu().numpy()
)
concat_val_metrics, _ = metrics_celeba(concat_val_y, concat_val_logits, concat_threshold)
concat_val_predictions = candidate_prediction_frame(
    concat_val_keys, concat_val_y, concat_val_codes, concat_val_logits, concat_threshold
)
concat_lcc_logits, concat_lcc_y, concat_lcc_keys, concat_lcc_codes, lcc_spatial, lcc_frequency = model_logits_concat(
    concat_model, lcc_loader, device, use_amp, return_features=True
)
assert_ordered_keys(concat_lcc_keys, resources["lcc_valid_paths"], "Phase B LCC")
concat_lcc_scores = binary_score(torch.as_tensor(concat_lcc_logits)).cpu().numpy()
concat_lcc_metrics = metrics_binary(
    concat_lcc_y, concat_lcc_scores, concat_threshold,
    coverage=lcc_coverage, candidate_n=lcc_candidate_n
)
concat_lcc_predictions = lcc_prediction_frame(lcc_frame, concat_lcc_keys, concat_lcc_logits)
concat_lcc_valid = concat_lcc_predictions["status"].astype(str).eq("VALID")
concat_lcc_predictions["locked_logit_threshold"] = concat_threshold
concat_lcc_predictions["pred_real"] = np.where(
    concat_lcc_valid, concat_lcc_predictions["pad_logit_score"] >= concat_threshold, np.nan
)
concat_lcc_predictions["correct_binary"] = np.where(
    concat_lcc_valid, concat_lcc_predictions["pred_real"] == (concat_lcc_predictions["label"] == 0), np.nan
)

val_ablation, val_rows, val_diag = branch_diagnostics(
    concat_model, concat_val_y, concat_val_keys, val_spatial, val_frequency,
    concat_threshold, device, resources["val_keys"]
)
lcc_ablation, lcc_rows, lcc_diag = branch_diagnostics(
    concat_model, concat_lcc_y, concat_lcc_keys, lcc_spatial, lcc_frequency,
    concat_threshold, device, resources["lcc_valid_paths"]
)
for row in val_rows:
    row["domain"] = "celeba_micro_val"
for row in lcc_rows:
    row["domain"] = "lcc_external_dev"
branch_rows = val_rows + lcc_rows
branch_ablation = {"val": val_ablation, "lcc": lcc_ablation}
feature_diagnostics = {
    "val": val_diag,
    "lcc": lcc_diag,
    "frozen_weights": True,
    "diagnostic_training": False,
    "frequency_shuffle": "deterministic label-independent Sattolo derangement; zero fixed points; seed and permutation SHA256 recorded",
}

concat_best_path = out_dir / "concat_best.pth"
concat_last_path = out_dir / "concat_checkpoint_last.pth"
torch.save({
    "state_dict": concat_best_state, "selected_global_epoch": int(concat_best_epoch),
    "validation_acer": float(concat_best_acer),
}, concat_best_path)
torch.save({"state_dict": concat_last_state, "completed_phase_epochs": 4}, concat_last_path)
concat_checkpoint_sha = sha256_file(concat_best_path)
concat_val_predictions.to_csv(out_dir / "concat_val_predictions.csv", index=False)
concat_lcc_predictions.to_csv(out_dir / "concat_lcc_predictions.csv", index=False)
pd.DataFrame(concat_history).to_csv(out_dir / "concat_training_history.csv", index=False)
pd.DataFrame(branch_rows).to_csv(out_dir / "branch_ablation.csv", index=False)

params = {
    "frequency_branch": count_params(concat_model.frequency_encoder),
    "fusion_head": count_params(concat_model.fusion),
    "total_trainable_phase_a": count_params(frequency_model),
    "total_trainable_phase_b": count_params(concat_model.frequency_encoder) + count_params(concat_model.fusion),
    "stage1_spatial_frozen_params": count_params(concat_model.spatial),
}
summary = make_candidate_summary(
    resources, stage1_sha, freq_val_metrics, freq_lcc_metrics,
    concat_val_metrics, concat_lcc_metrics, branch_ablation, feature_diagnostics, params
)
for branch, threshold in [("frequency_only", freq_threshold), ("concat", concat_threshold)]:
    summary[branch]["val"]["locked_logit_threshold"] = float(threshold)
    summary[branch]["lcc"]["locked_logit_threshold"] = float(threshold)
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")
(out_dir / "resource_fingerprints.json").write_text(json.dumps(summary["resource_hashes"], indent=2) + "\n")

config = {
    "schema": "pad_micro_candidate_config_v1",
    "candidate_id": CANDIDATE_ID, "candidate_name": CANDIDATE_NAME,
    "representation": CANDIDATE, "seed": SEED,
    "phase_a_budget": {
        "max_epochs": 5, "patience": 2, "earliest_early_stop_epoch": 3,
        "batch_size": BATCH_SIZE, "amp": use_amp, "optimizer": "AdamW",
        "lr": 3e-4, "weight_decay": 1e-4, "label_smoothing": 0.1, "grad_clip": 5.0,
    },
    "phase_b_budget": {
        "B1_epochs": 2, "B2_epochs": 2, "spatial_frozen": True,
        "frequency_lr_B2": 1e-4, "fusion_lr": 3e-4,
        "label_smoothing": 0.1, "grad_clip": 5.0,
    },
    "resource_hashes": summary["resource_hashes"],
    "threshold_policy": "calibrate separately on CelebA micro-Val; lock before LCC",
    "m4_coefficients_uv": M4_COEFFICIENTS if CANDIDATE_ID == "M4" else None,
    "m4_normalization_file": "m4_train_only_normalization_stats.json" if CANDIDATE_ID == "M4" else None,
}
(out_dir / "config.json").write_text(json.dumps(config, indent=2, allow_nan=False) + "\n")
(out_dir / "concat_meta.json").write_text(json.dumps({
    "schema": "pad_micro_concat_meta_v1", "candidate_id": CANDIDATE_ID,
    "checkpoint_sha256": concat_checkpoint_sha,
    "selected_global_epoch": int(concat_best_epoch),
    "locked_validation_logit_threshold": float(concat_threshold),
    "threshold_source": "CelebA micro-Val only",
    "resource_hashes": summary["resource_hashes"],
    "spatial_frozen_all_phase_b": True,
    "concat_dims": [256, 64, 320, 128, 3],
}, indent=2, allow_nan=False) + "\n")
(out_dir / "feature_diagnostics.json").write_text(json.dumps(feature_diagnostics, indent=2, allow_nan=False) + "\n")
metrics_rows = []
for branch, domains in [
    ("frequency_only", {"val": freq_val_metrics, "lcc": freq_lcc_metrics}),
    ("concat", {"val": concat_val_metrics, "lcc": concat_lcc_metrics}),
]:
    for domain, metric in domains.items():
        metrics_rows.append({"branch": branch, "domain": domain, **metric})
pd.DataFrame(metrics_rows).to_csv(out_dir / "metrics.csv", index=False)
result_zip = out_dir / f"micro_{CANDIDATE_ID.lower()}_{CANDIDATE_NAME}_results.zip"
zip_result_dir(out_dir, result_zip)
print("Candidate:", CANDIDATE_ID, CANDIDATE_NAME)
print("Stage 1 checkpoint SHA verified:", stage1_sha)
print("Frequency micro-Val:", freq_val_metrics)
print("Frequency LCC external-dev, Val threshold locked:", freq_lcc_metrics)
print("Concat micro-Val:", concat_val_metrics)
print("Concat LCC external-dev, Val threshold locked:", concat_lcc_metrics)
print("Branch diagnostics saved; no candidate self-promotes.")
print("Result ZIP:", result_zip)


Verified raw image paths before training: CelebA selected=7000, LCC candidates=3766
=== MICRO SEARCH PREFLIGHT ===
candidate: M4 block_dct_8x8
Stage 0 resource directory: /kaggle/input/datasets/maithanhphuong/smartface-pad-lcc-scrfd-cache-v1/stage0
Stage 1 result directory: /kaggle/input/datasets/maithanhphuong/smartface-pad-micro-stage1-v1/stage1_shared_micro_e1
Stage 1 checkpoint path/SHA: /kaggle/input/datasets/maithanhphuong/smartface-pad-micro-stage1-v1/stage1_shared_micro_e1/micro_e1_v1_best.pth 4fc6a179213a210a95950f90714276f316e173b596842fad8e7e6ccd833ab7e2
seed / diagnostic shuffle seed: 42 20260929
train_n / val_n: 5000 / 2000
train class counts: {1: 2314, 2: 1048, 0: 1638}
val class counts: {0: 659, 1: 920, 2: 421}
parent E1 manifest SHA: 7799486ebcae0babe09e061579ac05db7c1dc61e10746be0755bcddc10b2df9e
micro manifest SHA: 7c1eed4abaab7ee193758ea4bf4a0c36b98e43b5fb8d84c5c0e7d424607a0aa3
LCC manifest/cache SHA: af864ab57bf26af800ceac46c00d9724c8c60f5d0c63037394a1d8708a35abe5 2

/tmp/ipykernel_23/683733583.py:573: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  freq_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_23/683733583.py:368: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/tmp/ipykernel_23/683733583.py:276: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


{'phase': 'A', 'epoch': 1, 'train_loss': 1.0481202557682991, 'val_acer': 0.3254563950758103, 'val_auc': 0.7210923381753702, 'val_threshold': -0.6266975998878479, 'seconds': 30.938661098480225}


/tmp/ipykernel_23/683733583.py:368: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/tmp/ipykernel_23/683733583.py:276: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


{'phase': 'A', 'epoch': 2, 'train_loss': 0.9572206482291221, 'val_acer': 0.2574755097491397, 'val_auc': 0.8166102573329305, 'val_threshold': -0.6482547223567963, 'seconds': 26.009222984313965}


...

{'phase': 'B2', 'phase_epoch': 1, 'global_epoch': 3, 'train_loss': 0.45490573942661283, 'val_acer': 0.020650229314974556, 'val_auc': 0.9975942579032475, 'val_threshold': -0.8057201504707336, 'seconds': 25.678173780441284}


/tmp/ipykernel_23/683733583.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/tmp/ipykernel_23/683733583.py:298: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


{'phase': 'B2', 'phase_epoch': 2, 'global_epoch': 4, 'train_loss': 0.4546275772154331, 'val_acer': 0.020290386423738767, 'val_auc': 0.9976587580441294, 'val_threshold': -0.7435654103755951, 'seconds': 25.359928607940674}


/tmp/ipykernel_23/683733583.py:298: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/tmp/ipykernel_23/683733583.py:298: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


Candidate: M4 block_dct_8x8
Stage 1 checkpoint SHA verified: 4fc6a179213a210a95950f90714276f316e173b596842fad8e7e6ccd833ab7e2
Frequency micro-Val: {'n_scored': 2000, 'candidate_n': 2000, 'detector_coverage': 1.0, 'apcer': 0.2117822520507084, 'bpcer': 0.15174506828528073, 'acer': 0.18176366016799456, 'hter': 0.18176366016799456, 'auc': 0.8938825576908496, 'accuracy': 0.808, 'locked_logit_threshold': -1.2943655252456665, 'accuracy_3class': 0.6585}
Frequency LCC external-dev, Val threshold locked: {'n_scored': 3762, 'candidate_n': 3766, 'detector_coverage': 0.9989378651088688, 'apcer': 0.4017195375037059, 'bpcer': 0.7043701799485861, 'acer': 0.5530448587261461, 'hter': 0.5530448587261461, 'auc': 0.42098183289802504, 'accuracy': 0.5669856459330144, 'locked_logit_threshold': -1.2943655252456665}
Concat micro-Val: {'n_scored': 2000, 'candidate_n': 2000, 'detector_coverage': 1.0, 'apcer': 0.02237136465324385, 'bpcer': 0.018209408194233688, 'acer': 0.020290386423738767, 'hter': 0.0202903864237